# Renewable Generation Data Download — ENTSO-E
**TFG Arnau Ropero — GCIS Degree, UAB**

This notebook downloads hourly solar and wind generation data for peninsular Spain (2023–2025) via the ENTSO-E Transparency Platform API.

**Source:** ENTSO-E Transparency Platform (`transparency.entsoe.eu`)  
**Output:** `renewable_generation_ES_2023-2025.csv`

## 0. Install dependencies

In [ ]:
# Run this cell only the first time
!pip install entsoe-py pandas

## 1. Imports and configuration

In [ ]:
import pandas as pd
from entsoe import EntsoePandasClient

# ── CONFIGURATION ─────────────────────────────────────────────────
ENTSOE_API_KEY = "9bfc695e-472d-4a7f-bce1-28fc2d81f552"
COUNTRY        = "ES"                         # Peninsular Spain
OUTPUT_FILE    = "renewable_generation_ES_2023-2025.csv"

# Time range — same as the Open-Meteo meteorological dataset
START = pd.Timestamp("2023-01-01", tz="Europe/Madrid")
END   = pd.Timestamp("2025-06-01", tz="Europe/Madrid")  # exclusive
# ──────────────────────────────────────────────────────────────────

print(f"Configuration OK")
print(f"  Country: {COUNTRY}")
print(f"  Start:   {START.date()}")
print(f"  End:     {END.date()}")
print(f"  Output:  {OUTPUT_FILE}")

## 2. Data download

The `EntsoePandasClient` library automatically handles long time ranges (makes multiple internal calls year by year). May take 1–2 minutes.

In [ ]:
client = EntsoePandasClient(api_key=ENTSOE_API_KEY)

print("Downloading actual generation data by technology...")
print("(may take 1-2 minutes)")

df_raw = client.query_generation(
    COUNTRY,
    start=START,
    end=END,
    psr_type=None   # all technologies; will be filtered in the next cell
)

print(f"\nData successfully retrieved!")
print(f"  Records: {len(df_raw)}")
print(f"  Available technologies ({len(df_raw.columns)}):")
for col in df_raw.columns:
    print(f"    {col}")

## 3. Raw DataFrame inspection

In [ ]:
df_raw.head(10)

## 4. Filtering: solar and wind

In [ ]:
cols = df_raw.columns.tolist()

def col_str(c):
    return str(c).lower()

# Identify solar and wind columns
solar_cols = [c for c in cols if "solar" in col_str(c) or "photovoltaic" in col_str(c)]
wind_cols  = [c for c in cols if "wind" in col_str(c)]

print(f"Solar columns identified:  {solar_cols}")
print(f"Wind columns identified:   {wind_cols}")

In [ ]:
# Build the final DataFrame
df_out = pd.DataFrame(index=df_raw.index)

if solar_cols:
    df_out["solar_MW"] = df_raw[solar_cols].sum(axis=1)
else:
    print("WARNING: No solar columns found. Check column names above.")
    df_out["solar_MW"] = float("nan")

if wind_cols:
    df_out["wind_MW"] = df_raw[wind_cols].sum(axis=1)
else:
    print("WARNING: No wind columns found. Check column names above.")
    df_out["wind_MW"] = float("nan")

# Drop rows where all values are NaN
df_out = df_out.dropna(how="all")
df_out.index.name = "timestamp"
df_out = df_out.reset_index()

print(f"Final DataFrame: {len(df_out)} records")
df_out.head(10)

In [ ]:
# Convert timestamp to hourly (mean of each hour)
df_out["timestamp"] = pd.to_datetime(df_out["timestamp"], utc=True)
df_hourly = df_out.set_index("timestamp").resample("1h").mean()
df_hourly = df_hourly.reset_index()

print(f"Original records (15 min):   {len(df_out)}")
print(f"Aggregated records (hourly): {len(df_hourly)}")
df_hourly.head(5)

## 5. Statistics and validation

In [ ]:
print("=" * 50)
print("STATISTICS")
print("=" * 50)
print(f"Time range: {df_out['timestamp'].min()} -> {df_out['timestamp'].max()}")
print(f"Total records: {len(df_out)}")
print()
print(f"Solar:")
print(f"  Maximum:        {df_out['solar_MW'].max():.0f} MW")
print(f"  Mean:           {df_out['solar_MW'].mean():.0f} MW")
print(f"  Missing values: {df_out['solar_MW'].isna().sum()}")
print()
print(f"Wind:")
print(f"  Maximum:        {df_out['wind_MW'].max():.0f} MW")
print(f"  Mean:           {df_out['wind_MW'].mean():.0f} MW")
print(f"  Missing values: {df_out['wind_MW'].isna().sum()}")

In [ ]:
# Validation: expected number of records after hourly aggregation
# 2023-01-01 to 2025-05-31 = ~21,168 hours
expected = 21168
actual = len(df_hourly)
difference = abs(expected - actual)

print(f"Expected records (approx): {expected}")
print(f"Obtained records:          {actual}")
print(f"Difference:                {difference}")

if difference < 200:
    print("\nOK — Record count is consistent.")
else:
    print("\nWARNING — Large difference, there may be data gaps.")

## 6. Save CSV

In [ ]:
df_hourly.to_csv(OUTPUT_FILE, index=False)
print(f"File saved: {OUTPUT_FILE}")
print("Ready to merge with the Open-Meteo meteorological dataset!")